# Apache Spark Data Processing

## Step 1: Import Required Libraries

In this step, we import the required PySpark modules for creating a Spark session,
performing DataFrame operations, handling data types, and applying aggregation functions.

In [3]:
# Import SparkSession to create the entry point of the Spark application
from pyspark.sql import SparkSession

# Import commonly used SQL functions
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    min,
    max,
    when,
    to_date,
    desc
)

# Import data types (used later for schema modifications if required)
from pyspark.sql.types import *

## Step 2: Create a Spark Session

A SparkSession is the entry point for working with DataFrames and Spark SQL.
Every Spark application starts by creating a Spark session.

In [4]:
# Create a SparkSession
# appName() gives a name to the Spark application

spark = (
    SparkSession.builder
    .appName("Week5_Spark_Data_Processing")
    .getOrCreate()
)

# Display confirmation
print("Spark Session Created Successfully!")

Spark Session Created Successfully!


## Step 3: Load the Dataset

The dataset is stored in the `data` folder.

We use:
- header=True → First row contains column names
- inferSchema=True → Spark automatically detects data types

In [5]:
# Load the CSV dataset into a Spark DataFrame

df = spark.read.csv(
    "../data/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


## Step 4: Explore the Dataset

Before performing any transformations, it is important to understand the dataset.

We will:
- Display sample records
- View the schema
- Count rows and columns

In [6]:
# Display the first five rows
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [7]:
# Print the schema of the DataFrame
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [8]:
# Count the total number of rows
print("Total Rows:", df.count())

# Count the total number of columns
print("Total Columns:", len(df.columns))

Total Rows: 9994
Total Columns: 21


In [9]:
# Display all column names
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Step 5: Data Cleaning

Data cleaning is an essential step before performing any analysis.

In this section, we will:

- Remove duplicate records.
- Check for missing (null) values.
- Handle missing values if present.

This ensures that the dataset is clean and suitable for further transformations and analysis.

In [10]:
# Count the total number of records before removing duplicates
rows_before = df.count()

# Remove duplicate rows from the DataFrame
df = df.dropDuplicates()

# Count the total number of records after removing duplicates
rows_after = df.count()

# Display the results
print(f"Rows before removing duplicates : {rows_before}")
print(f"Rows after removing duplicates  : {rows_after}")
print(f"Duplicate rows removed          : {rows_before - rows_after}")

Rows before removing duplicates : 9994
Rows after removing duplicates  : 9994
Duplicate rows removed          : 0


In [11]:
# Import required functions
from pyspark.sql.functions import col, when, count

# Count the number of null values in each column
null_counts = df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in df.columns
])

# Display null counts
null_counts.show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [12]:
# Fill missing values in numeric columns
df = df.na.fill({
    "Sales": "0",
    "Quantity": "0",
    "Discount": "0",
    "Profit": 0
})

# Fill missing values in string columns
df = df.na.fill({
    "Category": "Unknown",
    "Region": "Unknown"
})

print("Missing values handled successfully.")

Missing values handled successfully.


In [13]:
# Verify that missing values have been handled
df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



## Step 6: Data Transformation

After cleaning the dataset, the next step is to transform the data into a suitable format for analysis.

In this section, we will:

- Rename column names for better readability.
- Convert data types to their appropriate formats.
- Convert date columns to `DateType`.
- Verify the updated schema.

### Rename Columns

Some column names contain spaces, making them inconvenient to reference in Spark queries.

We rename them using underscores for better readability.

In [14]:
# Rename columns containing spaces

df = (
    df.withColumnRenamed("Order Date", "Order_Date")
      .withColumnRenamed("Ship Date", "Ship_Date")
      .withColumnRenamed("Ship Mode", "Ship_Mode")
      .withColumnRenamed("Customer ID", "Customer_ID")
      .withColumnRenamed("Customer Name", "Customer_Name")
      .withColumnRenamed("Postal Code", "Postal_Code")
      .withColumnRenamed("Product ID", "Product_ID")
      .withColumnRenamed("Sub-Category", "Sub_Category")
      .withColumnRenamed("Product Name", "Product_Name")
)

print("Columns renamed successfully.")

Columns renamed successfully.


In [15]:
# Display updated column names
print(df.columns)

['Row ID', 'Order ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


### Cast Numeric Columns

Some numeric columns were loaded as strings because Spark could not infer their data types correctly.

We convert them into appropriate numeric data types.

In [16]:
from pyspark.sql.types import DoubleType, IntegerType

# Convert columns to appropriate data types

df = (
    df.withColumn("Sales", col("Sales").cast(DoubleType()))
      .withColumn("Quantity", col("Quantity").cast(IntegerType()))
      .withColumn("Discount", col("Discount").cast(DoubleType()))
)

### Convert Date Columns

The Order Date and Ship Date columns are currently stored as strings.

We convert them into Spark's `DateType` to enable date-based operations.

In [17]:
# Convert string dates to DateType

df = (
    df.withColumn("Order_Date", to_date(col("Order_Date"), "M/d/yyyy"))
      .withColumn("Ship_Date", to_date(col("Ship_Date"), "M/d/yyyy"))
)

print("Date columns converted successfully.")

Date columns converted successfully.


In [18]:
# Display updated schema
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = false)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = false)



In [19]:
# Display first five rows after transformation
df.show(5)

+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-----------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID| Customer_Name|    Segment|      Country|       City|         State|Postal_Code| Region|     Product_ID|       Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-----------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|   222|CA-2015-169397|2015-12-24|2015-12-27|   First Class|   JB-15925|Joni Blumstein|   Consumer|United States|     Dublin|          Ohio|      43017|   East|OFF-BI-10002852|Office Supplies|     Binders|Ibico 